# BudgetBuddy — train on SageMaker, serve from Lambda

**Goal:** you give it a prompt, it guesses how many tokens (input + output) Llama 3.3 70B will use.

## The whole thing in one picture

```
TRAINING (you run this now and then)                 PREDICTING (runs every time a user asks)

 your laptop / Studio notebook                        browser (Amplify site)
        | 1. upload numbers (CSV)                            | 6. POST {"prompt": "..."}
        v                                                    v
   +---------+   2. "go train"   +------------+        +-------------+
   |   S3    | <---------------- | SageMaker  |        | API Gateway |
   | bucket  | ----------------> | training   |        +-------------+
   |         |   3. reads CSV    | job        |              | 7. passes it on
   |         | <---------------- | (CatBoost) |              v
   |         |   4. writes       +------------+        +-------------+
   |         |      model.tar.gz                       |   Lambda    |
   |         |                                         | (CatBoost)  |
   |         | --------------------------------------> +-------------+
   +---------+   5./8. Lambda downloads model.tar.gz         | 9. {"predicted_total_tokens": 412}
                                                             v  back to the browser
```

## The same thing in words

Think of it as a **kitchen**:

- **S3 is the fridge.** It just stores files. It never runs anything.
- **SageMaker is a chef you rent by the minute.** You say "train". AWS switches on a computer, the chef takes the data *out of the fridge*, trains the model, puts the finished model *back in the fridge* as one file called `model.tar.gz`, and then the computer is switched off. You only pay for those minutes.
- **Lambda is the waiter.** It is asleep until a request arrives. When it wakes up it grabs `model.tar.gz` from the fridge, loads it into memory, and answers. While it stays awake it keeps the model in memory, so later answers are fast.
- **API Gateway is the front door.** It gives the Lambda a public URL.
- **Amplify hosts the website** that calls that URL.

**Important:** SageMaker and Lambda never talk to each other. They only share the fridge (S3).

**"The weights" = `model.tar.gz`.** For CatBoost this is not a fine-tuned neural network — it is a set of decision trees built from scratch on your data. It is tiny (a few MB), which is why a Lambda can run it.

**How does Lambda pick up a *new* model?** The Lambda always reads the same address: `models/current/model.tar.gz`. When you are happy with a new training run you copy its file to that address ("promote"), then poke the Lambda so it restarts and downloads the new file. That is section 6.

## 0. Settings — fill these in

You need three things before running anything (section 8 explains how to create them):

1. AWS credentials on this machine (`aws configure`).
2. An S3 bucket.
3. A SageMaker **execution role** — the permission slip that lets the rented chef open your fridge.

In [ ]:
import os, sys, json, tarfile
import boto3
import pandas as pd

REGION   = "ap-southeast-2"
BUCKET   = "budget-buddy-datasets"                                  # your S3 bucket name
ROLE_ARN = "arn:aws:iam::555108715183:role/service-role/AmazonSageMakerAdminIAMExecutionRole"  # your SageMaker execution role
LAMBDA_NAME = "budgetbuddy-predict"                                 # must match template.yaml

DATA_KEY_PREFIX = "data"                       # s3://BUCKET/data/train/..., /validation/...
CURRENT_MODEL_KEY = "models/current/model.tar.gz"   # the address the Lambda always reads

os.environ["AWS_DEFAULT_REGION"] = REGION
s3 = boto3.client("s3", region_name=REGION)
os.makedirs("lambda_fn", exist_ok=True)
os.makedirs("build", exist_ok=True)

## 1. Turn prompts into numbers (features)

CatBoost cannot read text. It reads a table of numbers. So we turn each prompt into numbers like "how many characters", "how many words", "does it contain code".

This lives in its own file because **the Lambda must do exactly the same thing** to a new prompt. One file, used in both places = they can never disagree.

In [ ]:
%%writefile lambda_fn/features.py
"""Turn a prompt (text) into numbers. Used by BOTH training and the Lambda, so they always agree."""
import re

# The order here is the column order of the training CSV. Never reorder without retraining.
FEATURE_NAMES = [
    "n_chars", "n_words", "n_lines", "n_sentences", "n_question_marks",
    "n_digits", "n_code_symbols", "n_code_fences", "n_code_keywords",
    "avg_word_len", "has_tests", "asks_for_short", "asks_for_long",
]

_CODE_KEYWORDS = re.compile(r"\b(def|class|import|return|function|assert|SELECT|for|while)\b")
_SHORT = re.compile(r"\b(brief|briefly|short|concise|one word|one sentence|yes or no|summari[sz]e)\b", re.I)
_LONG = re.compile(r"\b(explain|detailed|in detail|essay|step by step|write a|implement|story)\b", re.I)


def extract_features(prompt):
    text = prompt if isinstance(prompt, str) else ""
    words = text.split()
    return [
        len(text),
        len(words),
        text.count("\n") + 1,
        len(re.findall(r"[.!?]+", text)),
        text.count("?"),
        sum(c.isdigit() for c in text),
        sum(text.count(c) for c in "{}()[]=<>;_"),
        text.count("```"),
        len(_CODE_KEYWORDS.findall(text)),
        (sum(len(w) for w in words) / len(words)) if words else 0.0,
        int("assert" in text or "test" in text.lower()),
        int(bool(_SHORT.search(text))),
        int(bool(_LONG.search(text))),
    ]

In [ ]:
sys.path.insert(0, "lambda_fn")
from features import extract_features, FEATURE_NAMES

extract_features("Write a Python function that reverses a string. Explain it step by step.")

## 2. Build the training files

- **Target (the answer we want to predict)** = `input_size` + `output_mean`. The dataset ran every prompt 10 times; `output_mean` is the average output length.
- The dataset already has a `split` column (`train` / `validate` / `test`), so we use it.
- The SageMaker CatBoost algorithm wants a CSV with **no header** and **the target in the first column**.

We only write numbers to these files. The prompt text itself is **not** uploaded to S3 (some ShareGPT prompts contain real names, phone numbers and emails).

In [ ]:
df = pd.read_csv("dataset_castillo.csv")
df["target"] = df["input_size"] + df["output_mean"]

X = pd.DataFrame([extract_features(t) for t in df["prompt_text"]], columns=FEATURE_NAMES)
table = pd.concat([df["target"], X], axis=1)   # target first, then the features

local_files = {}
for split, folder in {"train": "train", "validate": "validation", "test": "test"}.items():
    path = f"build/{folder}.csv"
    table[df["split"] == split].to_csv(path, header=False, index=False)
    local_files[folder] = path
    print(folder, (df["split"] == split).sum(), "rows ->", path)

## 3. Laptop → S3: put the data in the fridge

The training job will be pointed at `s3://BUCKET/data/` and expects two folders inside it: `train/` and `validation/`. The test file goes somewhere else so training never sees it.

In [ ]:
s3.upload_file(local_files["train"],      BUCKET, f"{DATA_KEY_PREFIX}/train/data.csv")
s3.upload_file(local_files["validation"], BUCKET, f"{DATA_KEY_PREFIX}/validation/data.csv")
s3.upload_file(local_files["test"],       BUCKET, "eval/test.csv")
print("uploaded to", f"s3://{BUCKET}/{DATA_KEY_PREFIX}/")

## 4. S3 → SageMaker → S3: run the training job

This cell does **not** train on your laptop. It sends a request to AWS that says:

- use the built-in **CatBoost regression** algorithm (`catboost-regression-model`, the same one you see in SageMaker Studio / JumpStart),
- on one rented computer (`ml.m5.xlarge`),
- read the data from `s3://BUCKET/data/`,
- write the result under `s3://BUCKET/models/`,
- and use `ROLE_ARN` as the permission slip.

Then it waits and prints the logs. Expect a few minutes, most of it the computer starting up.

In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import InputData, Compute, OutputDataConfig
from sagemaker.core.jumpstart.configs import JumpStartConfig

trainer = ModelTrainer.from_jumpstart_config(
    jumpstart_config=JumpStartConfig(model_id="catboost-regression-model"),
    compute=Compute(instance_type="ml.m5.xlarge", instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{BUCKET}/models/"),
    role=ROLE_ARN,
    base_job_name="budgetbuddy-catboost",
    # hyperparameters={"iterations": "1000", "learning_rate": "0.05", "depth": "6"},  # optional tuning knobs
)

trainer.train(
    input_data_config=[
        InputData(channel_name="training", data_source=f"s3://{BUCKET}/{DATA_KEY_PREFIX}/", content_type="text/csv")
    ],
    wait=True,
)

Now ask SageMaker where it put the finished model. That S3 address is the "weights".

In [ ]:
job_name = trainer._latest_training_job.training_job_name
job = boto3.client("sagemaker", region_name=REGION).describe_training_job(TrainingJobName=job_name)

MODEL_S3_URI = job["ModelArtifacts"]["S3ModelArtifacts"]   # s3://BUCKET/models/<job-name>/output/model.tar.gz
print(job["TrainingJobStatus"], MODEL_S3_URI)

## 5. Evaluate: is the model any good?

We download the model, open it, and test it on the `test` rows it has never seen.

- **MAE** = on average, how many tokens off is the guess?
- **Baseline** = a dumb model that always guesses the average. Your model must beat this or it has learned nothing.

The file `model_io.py` is the "open the tar and load CatBoost" helper. The Lambda uses the same file.

In [ ]:
%%writefile lambda_fn/model_io.py
"""Open the model.tar.gz that SageMaker produced and load the CatBoost model inside it."""
import os
import tarfile

from catboost import CatBoostRegressor

_SKIP = (".py", ".json", ".txt", ".md", ".sh", ".yaml", ".yml", ".csv", ".tsv")


def load_model_from_tar(tar_path, extract_dir):
    with tarfile.open(tar_path) as tar:
        tar.extractall(extract_dir)

    tried = []
    for root, _, files in os.walk(extract_dir):
        for name in files:
            if name.endswith(_SKIP):
                continue
            path = os.path.join(root, name)
            try:
                model = CatBoostRegressor()
                model.load_model(path)
                return model
            except Exception:
                tried.append(path)
    raise RuntimeError(f"No CatBoost model file found in {tar_path}. Tried: {tried}")

In [ ]:
from model_io import load_model_from_tar

model_key = MODEL_S3_URI.replace(f"s3://{BUCKET}/", "")
s3.download_file(BUCKET, model_key, "build/model.tar.gz")

with tarfile.open("build/model.tar.gz") as tar:
    print("files inside model.tar.gz:", tar.getnames())

model = load_model_from_tar("build/model.tar.gz", "build/model")

test = pd.read_csv("build/test.csv", header=None)
y_true, X_test = test[0], test.drop(columns=0)
y_pred = model.predict(X_test.values)

train_mean = pd.read_csv("build/train.csv", header=None)[0].mean()
print(f"model MAE:    {(y_true - y_pred).abs().mean():.1f} tokens")
print(f"baseline MAE: {(y_true - train_mean).abs().mean():.1f} tokens  (always guessing {train_mean:.0f})")

## 6. Promote: tell the Lambda to use the new model

Two steps:

1. **Copy** the new `model.tar.gz` to the fixed address the Lambda reads (`models/current/model.tar.gz`).
2. **Poke the Lambda.** A warm Lambda still has the *old* model in memory. Changing any setting on it forces a restart, and on restart it downloads the new file.

Only run this when section 5 looks good. (The second half needs the Lambda to exist — see section 8.)

In [ ]:
s3.copy_object(Bucket=BUCKET, Key=CURRENT_MODEL_KEY, CopySource={"Bucket": BUCKET, "Key": model_key})
print("current model is now", MODEL_S3_URI)

lam = boto3.client("lambda", region_name=REGION)
env = lam.get_function_configuration(FunctionName=LAMBDA_NAME)["Environment"]["Variables"]
env["MODEL_VERSION"] = job_name   # the value is just a label; changing it is what forces the restart
lam.update_function_configuration(FunctionName=LAMBDA_NAME, Environment={"Variables": env})
print("Lambda will load the new model on its next request")

## 7. The Lambda itself (S3 → Lambda → API Gateway)

Three small files:

- `app.py` — the code that runs on every request.
- `Dockerfile` — the recipe for the box the Lambda runs in. CatBoost is too big for a normal Lambda zip (250 MB limit), so we ship the Lambda as a **container image** instead (10 GB limit).
- `template.yaml` — the blueprint that tells AWS "create this Lambda, give it a public URL, and let it read my bucket".

What `app.py` does on a request:

1. First request after waking up ("cold start"): download `model.tar.gz` from S3 into `/tmp` and load it. This takes a few seconds.
2. Every request: prompt → `extract_features` → `model.predict` → JSON answer.

In [ ]:
%%writefile lambda_fn/app.py
"""The Lambda. API Gateway hands it a prompt, it hands back a predicted token count."""
import json
import os

import boto3

from features import extract_features
from model_io import load_model_from_tar

MODEL_BUCKET = os.environ["MODEL_BUCKET"]
MODEL_KEY = os.environ.get("MODEL_KEY", "models/current/model.tar.gz")

_model = None  # kept in memory between requests while the Lambda stays warm


def _get_model():
    global _model
    if _model is None:
        # /tmp is the only folder a Lambda is allowed to write to
        boto3.client("s3").download_file(MODEL_BUCKET, MODEL_KEY, "/tmp/model.tar.gz")
        _model = load_model_from_tar("/tmp/model.tar.gz", "/tmp/model")
    return _model


def _response(status, body):
    return {"statusCode": status, "headers": {"Content-Type": "application/json"}, "body": json.dumps(body)}


def handler(event, context):
    try:
        prompt = json.loads(event.get("body") or "{}").get("prompt")
    except json.JSONDecodeError:
        return _response(400, {"error": "Body must be JSON"})
    if not isinstance(prompt, str) or not prompt:
        return _response(400, {"error": 'Send JSON like {"prompt": "your text"}'})

    prediction = float(_get_model().predict([extract_features(prompt)])[0])
    return _response(200, {"predicted_total_tokens": max(0, round(prediction))})

In [ ]:
%%writefile lambda_fn/Dockerfile
FROM public.ecr.aws/lambda/python:3.12
RUN pip install --no-cache-dir catboost==1.2.10 boto3
COPY features.py model_io.py app.py ${LAMBDA_TASK_ROOT}/
CMD ["app.handler"]

In [ ]:
%%writefile template.yaml
AWSTemplateFormatVersion: "2010-09-09"
Transform: AWS::Serverless-2016-10-31
Description: BudgetBuddy - Lambda + API Gateway that serve token predictions

Parameters:
  ModelBucket:
    Type: String
    Description: The S3 bucket that holds models/current/model.tar.gz

Resources:
  PredictFunction:
    Type: AWS::Serverless::Function
    Properties:
      FunctionName: budgetbuddy-predict
      PackageType: Image
      MemorySize: 1024
      Timeout: 30
      Environment:
        Variables:
          MODEL_BUCKET: !Ref ModelBucket
          MODEL_KEY: models/current/model.tar.gz
      Policies:
        - S3ReadPolicy:
            BucketName: !Ref ModelBucket
      Events:
        Predict:
          Type: HttpApi
          Properties:
            ApiId: !Ref Api
            Path: /predict
            Method: POST
    Metadata:
      Dockerfile: Dockerfile
      DockerContext: ./lambda_fn
      DockerTag: latest

  Api:
    Type: AWS::Serverless::HttpApi
    Properties:
      CorsConfiguration:
        AllowOrigins: ["*"]   # tighten to your Amplify URL once you have it
        AllowMethods: [POST, OPTIONS]
        AllowHeaders: [content-type]

Outputs:
  PredictUrl:
    Value: !Sub "https://${Api}.execute-api.${AWS::Region}.amazonaws.com/predict"

## 8. How to deploy all of this

**Short answer:** use the **console** for the two one-time things that are easiest to click (the SageMaker role and Amplify), and **code** for everything else, so you can rebuild it with one command.

| Piece | How | Why |
|---|---|---|
| AWS login on your laptop | CLI: `aws configure` | one time |
| S3 bucket | CLI: one command | one time |
| SageMaker role | Console | one time, a few clicks |
| Training job | This notebook (section 4) | you re-run it whenever you retrain |
| Lambda + API Gateway + Lambda's permissions | **AWS SAM** (`template.yaml`) | "infrastructure as code": one command creates or updates all of it |
| Frontend | Amplify console, connected to your GitHub repo | it redeploys by itself on every push |

### Do it in this order

**A. One-time setup**

1. Install the [AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html), the [SAM CLI](https://docs.aws.amazon.com/serverless-application-model/latest/developerguide/install-sam-cli.html) and Docker Desktop.
2. `aws configure` — paste an access key for your IAM user and pick a region (use the same one everywhere).
3. Make the bucket (the name must be unique in the whole world):
   ```
   aws s3 mb s3://budgetbuddy-yourname-123 --region us-east-1
   ```
4. Make the SageMaker role: Console → **IAM** → Roles → Create role → AWS service → **SageMaker** → keep `AmazonSageMakerFullAccess` → name it `BudgetBuddySageMakerRole`. Then add S3 access to your bucket (attach `AmazonS3FullAccess` to get going; narrow it to your one bucket later). Copy the role's ARN.
5. Put the region, bucket name and role ARN into section 0.

**B. Train the first model**

6. Run sections 0 → 5 of this notebook.
7. Run only the **first half** of section 6 (the `copy_object` lines). The Lambda does not exist yet.

**C. Create the Lambda and the API**

8. Run the three `%%writefile` cells in section 7 so the files exist, then in a terminal in this folder:
   ```
   sam build
   sam deploy --guided
   ```
   It asks a few questions. Stack name: `budgetbuddy`. `ModelBucket`: your bucket name. Say **yes** to creating IAM roles and to creating the image repository. When it warns that the function has no authorization, say **yes** (see the warning below).
9. At the end it prints `PredictUrl`. Test it:
   ```
   curl -X POST <PredictUrl> -H "content-type: application/json" -d "{\"prompt\": \"Explain recursion\"}"
   ```

**D. Frontend**

10. Console → **Amplify** → Create new app → connect your GitHub repo → deploy. In your frontend code, call the API:
    ```js
    const res = await fetch(PREDICT_URL, {
      method: "POST",
      headers: { "content-type": "application/json" },
      body: JSON.stringify({ prompt }),
    });
    const { predicted_total_tokens } = await res.json();
    ```
11. Edit `AllowOrigins` in `template.yaml` to your Amplify URL and run `sam build && sam deploy` again.

**E. Every time you retrain**

12. Run sections 2 → 6. Section 6 swaps the model in. You do **not** redeploy the Lambda — its code did not change, only the file in S3 did.
    You only run `sam build && sam deploy` again when you change `features.py`, `app.py` or `template.yaml`. If you change `features.py`, retrain too.

### Things that will bite you

- **The URL is public.** Anyone who finds it can call it and you pay for the calls. Fine for a demo; add an API key, a Cognito authorizer or throttling before you share it.
- **Cold starts.** The first request after a quiet period takes several seconds while the Lambda downloads and loads the model.
- **Cost.** Training is a few cents per run. Lambda, API Gateway and S3 at hobby volume are close to free. Nothing here runs 24/7.
- **Same region everywhere.** Bucket, training job, Lambda and API must all be in one region.